# WoE sensitivity — число бинов, сглаживание и фон

Эксперимент для расширенной таблицы `features_table_geo.csv`. WoE здесь — логарифм отношения частоты признака среди известных месторождений к частоте в Unlabeled background. Это не вероятность месторождения и не сравнение с достоверными отрицательными примерами. Для корректной CV границы бинов и веса должны оцениваться только на train fold.

In [ ]:

from pathlib import Path
import json
import warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

TARGET = "is_deposit"
FOLD = "fold_id"
TOP_FRACTIONS = (0.01, 0.05, 0.10)
ALPHA = 0.5

def find_project_root():
    candidates = [Path.cwd(), *Path.cwd().parents]
    for candidate in candidates:
        if (candidate / "data" / "processed").exists():
            return candidate
    # Common location when notebook is opened from backend/notebooks
    candidate = Path("../../")
    if (candidate / "data" / "processed").exists():
        return candidate.resolve()
    raise FileNotFoundError(
        "Не найдена папка data/processed. Запустите ноутбук из репозитория "
        "или проверьте расположение данных."
    )

ROOT = find_project_root()
DATA_PATH = ROOT / "data" / "processed" / "features_table_geo.csv"
if not DATA_PATH.exists():
    raise FileNotFoundError(f"Не найден файл: {DATA_PATH}")

df = pd.read_csv(DATA_PATH)
NUMERIC_FEATURES = [
    c for c in ["elevation", "magnetic", "gravity",
                "dist_fault_km", "dist_contact_km"] if c in df.columns
]
CATEGORICAL_FEATURES = [
    c for c in ["lithology_group"] if c in df.columns
]
FEATURES = NUMERIC_FEATURES + CATEGORICAL_FEATURES

required = set(FEATURES + [TARGET, FOLD, "x", "y"])
missing = sorted(required - set(df.columns))
if missing:
    raise ValueError(f"В таблице отсутствуют обязательные колонки: {missing}")
if not set(df[TARGET].dropna().unique()).issubset({0, 1}):
    raise ValueError("is_deposit должен содержать только 0/1.")
if df[[TARGET, FOLD, "x", "y"]].isna().any().any():
    raise ValueError("Есть пропуски в target/fold_id/x/y.")
if df.duplicated(["x", "y"]).any():
    raise ValueError("Найдены дубли координат ячеек.")

print("Data:", DATA_PATH)
print("Shape:", df.shape)
print("Known deposits:", int(df[TARGET].sum()))
print("Unlabeled cells:", int((df[TARGET] == 0).sum()))
print("Numeric:", NUMERIC_FEATURES)
print("Categorical:", CATEGORICAL_FEATURES)
display(df.groupby(FOLD)[TARGET].agg(["size", "sum"]).rename(
    columns={"size": "cells", "sum": "known_deposits"}
))


Data: d:\Uni\Сборники\3 курс\Командная разработка\Удалённый репозиторий\data\processed\features_table_geo.csv
Shape: (35694, 13)
Known deposits: 11
Unlabeled cells: 35683
Numeric: ['elevation', 'magnetic', 'gravity', 'dist_fault_km', 'dist_contact_km']
Categorical: ['lithology_group']


,cells,known_deposits
fold_id,,
0,4874,1
1,7273,2
2,3359,2
3,17250,5
4,2938,1


## 1. WoE fit/transform с обучаемыми границами

In [13]:

def _safe_numeric_edges(series, n_bins):
    """Квантильные границы оцениваются только по train."""
    values = pd.to_numeric(series, errors="coerce").dropna()
    if values.empty:
        return np.array([-np.inf, np.inf], dtype=float)
    quantiles = np.unique(np.quantile(values, np.linspace(0, 1, n_bins + 1)))
    if len(quantiles) < 2:
        return np.array([-np.inf, np.inf], dtype=float)
    quantiles[0], quantiles[-1] = -np.inf, np.inf
    return quantiles.astype(float)


def fit_woe(train, features, n_bins=5, alpha=0.5, categorical=()):
    """
    WoE относительно Unlabeled-фона, не относительно истинных отрицательных.
    Числовые edges и все WoE-таблицы fit-ятся только на train.
    Пропуски числовых признаков получают отдельную категорию <missing>.
    """
    pos = train.loc[train[TARGET].eq(1)]
    unl = train.loc[train[TARGET].eq(0)]
    if len(pos) == 0 or len(unl) == 0:
        raise ValueError("В train должны быть Positive и Unlabeled ячейки.")
    if alpha <= 0:
        raise ValueError("alpha должен быть > 0.")

    models = {}
    for feature in features:
        if feature in categorical:
            p = pos[feature].fillna("<missing>").astype(str)
            u = unl[feature].fillna("<missing>").astype(str)
            categories = sorted(set(p.unique()) | set(u.unique()))
            p_counts, u_counts = p.value_counts(), u.value_counts()
            k = max(len(categories), 1)
            p_den = len(p) + alpha * k
            u_den = len(u) + alpha * k
            table = {
                category: float(np.log(
                    ((p_counts.get(category, 0) + alpha) / p_den) /
                    ((u_counts.get(category, 0) + alpha) / u_den)
                ))
                for category in categories
            }
            models[feature] = {
                "categorical": True, "edges": None, "table": table,
                "categories": categories
            }
        else:
            edges = _safe_numeric_edges(train[feature], n_bins)
            p_values = pd.to_numeric(pos[feature], errors="coerce")
            u_values = pd.to_numeric(unl[feature], errors="coerce")
            p_bins = pd.cut(p_values, bins=edges, include_lowest=True)
            u_bins = pd.cut(u_values, bins=edges, include_lowest=True)
            categories = list(p_bins.cat.categories)
            p_counts = p_bins.value_counts().reindex(categories, fill_value=0)
            u_counts = u_bins.value_counts().reindex(categories, fill_value=0)
            k = max(len(categories), 1)
            p_den = len(pos) + alpha * k
            u_den = len(unl) + alpha * k
            table = {
                category: float(np.log(
                    ((p_counts.loc[category] + alpha) / p_den) /
                    ((u_counts.loc[category] + alpha) / u_den)
                ))
                for category in categories
            }
            models[feature] = {
                "categorical": False, "edges": edges, "table": table,
                "categories": categories
            }
    return models


def score_woe(data, models):
    """Суммарный ranking score; это не вероятность."""
    score = pd.Series(0.0, index=data.index, dtype=float)
    for feature, model in models.items():
        if model["categorical"]:
            values = data[feature].fillna("<missing>").astype(str)
        else:
            values = pd.cut(
                pd.to_numeric(data[feature], errors="coerce"),
                bins=model["edges"], include_lowest=True
            )
        contribution = values.map(model["table"])
        contribution = pd.to_numeric(contribution, errors="coerce").fillna(0.0)
        score = score.add(contribution.astype(float))
    return score


def tie_aware_topk(y_true, scores, fractions=TOP_FRACTIONS):
    """
    Для score на границе top-K возвращает нижнюю/верхнюю/ожидаемую
    оценку числа Positive при случайном разрешении ничьих.
    """
    y = np.asarray(y_true, dtype=int)
    s = np.asarray(scores, dtype=float)
    if len(y) != len(s):
        raise ValueError("y_true и scores должны иметь одинаковую длину.")
    if not np.isfinite(s).all():
        raise ValueError("В scores есть NaN/inf.")
    n, n_pos = len(y), int(y.sum())
    rows = []
    for fraction in fractions:
        k = max(1, int(np.ceil(n * fraction)))
        cutoff = np.sort(s)[::-1][k - 1]
        above, tied = s > cutoff, s == cutoff
        found_above = int(y[above].sum())
        pos_tied = int(y[tied].sum())
        slots = max(k - int(above.sum()), 0)
        n_tied = int(tied.sum())
        expected = found_above + (
            slots * pos_tied / n_tied if n_tied else 0.0
        )
        found_max = found_above + min(slots, pos_tied)
        rows.append({
            "area_fraction": fraction, "top_k_cells": k,
            "cutoff_score": float(cutoff),
            "found_min": found_above, "found_max": found_max,
            "found_expected": expected,
            "precision_at_k_expected": expected / k,
            "success_rate_expected": expected / n_pos if n_pos else np.nan,
            "cells_tied_at_cutoff": n_tied,
            "positive_ties_at_cutoff": pos_tied,
        })
    return pd.DataFrame(rows)


def deposit_ranks_in_test_fold(test, scores):
    """Ранг месторождения среди ВСЕХ ячеек тестового фолда."""
    scores = pd.Series(np.asarray(scores, dtype=float), index=test.index)
    positive = test.loc[test[TARGET].eq(1),
                        [c for c in ["deposit_name", "x", "y", FOLD] if c in test]].copy()
    positive["score"] = scores.loc[positive.index]
    # percentile rank = доля ячеек со score >= score объекта (min rank)
    positive["rank_min_all_cells"] = positive["score"].map(
        lambda value: int((scores >= value).sum())
    )
    positive["rank_fraction_all_cells"] = (
        positive["rank_min_all_cells"] / len(test)
    )
    positive["score_percentile"] = positive["score"].rank(
        ascending=True, pct=True, method="average"
    )
    return positive.sort_values("rank_min_all_cells")


def run_spatial_oof(data, features, n_bins=5, alpha=0.5, categorical=()):
    rows, parts = [], []
    folds = sorted(data[FOLD].dropna().unique())
    for fold in folds:
        train = data.loc[data[FOLD].ne(fold)].copy()
        test = data.loc[data[FOLD].eq(fold)].copy()
        n_train_pos = int(train[TARGET].sum())
        n_test_pos = int(test[TARGET].sum())
        if n_train_pos == 0 or n_test_pos == 0:
            warnings.warn(
                f"Fold {fold}: train positives={n_train_pos}, "
                f"test positives={n_test_pos}; fold skipped."
            )
            continue
        model = fit_woe(train, features, n_bins, alpha, categorical)
        scores = score_woe(test, model)
        metrics = tie_aware_topk(test[TARGET], scores)
        metrics[FOLD] = fold
        metrics["test_cells"] = len(test)
        metrics["test_deposits"] = n_test_pos
        metrics["train_deposits"] = n_train_pos
        metrics["n_unique_scores"] = int(scores.nunique())
        rows.append(metrics)

        part = test[["x", "y", TARGET, FOLD]].copy()
        if "deposit_name" in test.columns:
            part["deposit_name"] = test["deposit_name"]
        part["score_oof"] = scores
        part["n_bins"] = n_bins
        part["alpha"] = alpha
        parts.append(part)
    return pd.concat(rows, ignore_index=True), pd.concat(parts, ignore_index=True)


## 2. Spatial OOF: сравнение числа бинов и сглаживания

Каждый fold выступает holdout-блоком. Границы квантилей и WoE-таблицы обучаются только на остальных фолдах. Для top-K учитываются ничьи на границе: выводятся минимальное, максимальное и ожидаемое число известных объектов.

Важно: среднее Success Rate по фолдам — описательная сводка, а не единая pooled-оценка. Всегда рассматривайте число Positive в каждом тестовом фолде.

In [14]:

results = []
for n_bins in [3,  5, 7]:
    for alpha in [0.1, 0.5, 1.0]:
        for fold in sorted(df[FOLD].dropna().unique()):
            train = df.loc[df[FOLD].ne(fold)].copy()
            test = df.loc[df[FOLD].eq(fold)].copy()
            if train[TARGET].sum() == 0 or test[TARGET].sum() == 0:
                warnings.warn(
                    f"Fold {fold} пропущен: train Positive={int(train[TARGET].sum())}, "
                    f"test Positive={int(test[TARGET].sum())}"
                )
                continue
            model = fit_woe(
                train, FEATURES, n_bins=n_bins, alpha=alpha,
                categorical=CATEGORICAL_FEATURES
            )
            scores = score_woe(test, model)
            metrics = tie_aware_topk(test[TARGET], scores)
            metrics[FOLD] = fold
            metrics["n_bins"] = n_bins
            metrics["alpha"] = alpha
            metrics["test_cells"] = len(test)
            metrics["test_deposits"] = int(test[TARGET].sum())
            metrics["n_unique_scores"] = int(scores.nunique())
            results.append(metrics)

cv = pd.concat(results, ignore_index=True) if results else pd.DataFrame()
display(cv.sort_values(["n_bins", "alpha", FOLD, "area_fraction"]))

if len(cv):
    summary = (
        cv.groupby(["n_bins", "alpha", "area_fraction"])
        .agg(
            mean_success_rate=("success_rate_expected", "mean"),
            std_success_rate=("success_rate_expected", "std"),
            mean_precision_at_k=("precision_at_k_expected", "mean"),
            mean_found_expected=("found_expected", "mean"),
            total_test_deposits=("test_deposits", "sum"),
            folds=("fold_id", "nunique"),
        ).reset_index()
    )
    display(summary)


,area_fraction,top_k_cells,cutoff_score,found_min,found_max,found_expected,precision_at_k_expected,success_rate_expected,cells_tied_at_cutoff,positive_ties_at_cutoff,fold_id,n_bins,alpha,test_cells,test_deposits,n_unique_scores
0,0.01,49,2.282475,0,0,0.0,0.000000,0.0,11,0,0,3,0.1,4874,1,344
1,0.05,244,1.138929,0,0,0.0,0.000000,0.0,29,0,0,3,0.1,4874,1,344
2,0.10,488,1.132197,1,1,1.0,0.002049,1.0,2,0,0,3,0.1,4874,1,344
3,0.01,73,2.281539,0,0,0.0,0.000000,0.0,8,0,1,3,0.1,7273,2,460
4,0.05,364,1.161897,0,0,0.0,0.000000,0.0,27,0,1,3,0.1,7273,2,460
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
130,0.05,863,1.129941,1,1,1.0,0.001159,0.2,33,0,3,7,1.0,17250,5,1773
131,0.10,1725,0.723718,1,1,1.0,0.000580,0.2,9,0,3,7,1.0,17250,5,1773
132,0.01,30,2.911976,0,0,0.0,0.000000,0.0,4,0,4,7,1.0,2938,1,1476
133,0.05,147,1.979448,0,0,0.0,0.000000,0.0,1,0,4,7,1.0,2938,1,1476


,n_bins,alpha,area_fraction,mean_success_rate,std_success_rate,mean_precision_at_k,mean_found_expected,total_test_deposits,folds
0,3,0.1,0.01,0.040000,0.089443,0.001156,0.200000,11,5
1,3,0.1,0.05,0.040000,0.089443,0.000232,0.200000,11,5
2,3,0.1,0.10,0.240000,0.433590,0.000526,0.400000,11,5
3,3,0.5,0.01,0.040000,0.089443,0.001156,0.200000,11,5
4,3,0.5,0.05,0.080000,0.178885,0.000463,0.400000,11,5
5,3,0.5,0.10,0.131948,0.187359,0.000338,0.451948,11,5
6,3,1.0,0.01,0.000000,0.000000,0.000000,0.000000,11,5
7,3,1.0,0.05,0.080000,0.178885,0.000463,0.400000,11,5
8,3,1.0,0.10,0.080000,0.178885,0.000232,0.400000,11,5
9,5,0.1,0.01,0.000000,0.000000,0.000000,0.000000,11,5


## 3. Сравнение с полной таблицей (только exploratory)

Оценка на всей таблице использует все известные месторождения при fit и потому не является validation. Она нужна только для просмотра итогового распределения score и WoE-таблиц.

In [15]:

full_models = fit_woe(
    df, FEATURES, n_bins=5, alpha=0.5,
    categorical=CATEGORICAL_FEATURES
)
df["exploratory_woe_score"] = score_woe(df, full_models)
display(tie_aware_topk(df[TARGET], df["exploratory_woe_score"]))

display(pd.DataFrame([
    {
        "feature": feature,
        "n_categories": len(model["table"]),
        "min_woe": min(model["table"].values()),
        "max_woe": max(model["table"].values()),
        "n_unique_scores": df[feature].nunique(dropna=False),
    }
    for feature, model in full_models.items()
]))

# Сохраняем результаты sensitivity для воспроизводимости
OUTPUT_DIR = ROOT / "data" / "processed" / "woe_sensitivity"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
cv.to_csv(OUTPUT_DIR / "sensitivity_cv_metrics.csv", index=False)
print("Saved to:", OUTPUT_DIR)


,area_fraction,top_k_cells,cutoff_score,found_min,found_max,found_expected,precision_at_k_expected,success_rate_expected,cells_tied_at_cutoff,positive_ties_at_cutoff
0,0.01,357,2.404252,2,2,2.0,0.005602,0.181818,6,0
1,0.05,1785,1.500871,4,4,4.0,0.002241,0.363636,7,0
2,0.10,3570,0.986229,6,6,6.0,0.001681,0.545455,15,0


,feature,n_categories,min_woe,max_woe,n_unique_scores
0,elevation,5,-0.592456,0.268462,779
1,magnetic,5,-1.686735,0.511050,35636
2,gravity,5,-0.587983,0.511050,35604
3,dist_fault_km,5,-0.587983,0.511190,35694
4,dist_contact_km,5,-0.587983,0.259595,35694
5,lithology_group,8,-0.897366,2.265103,8


Saved to: d:\Uni\Сборники\3 курс\Командная разработка\Удалённый репозиторий\data\processed\woe_sensitivity


## 4. Ограничения и правила интерпретации

- `is_deposit=0` — Unlabeled background, а не Negative.
- Границы бинов и WoE-таблицы fit-ятся только на train fold.
- Для неизвестных категорий и пропусков при scoring применяется нейтральный вклад 0; их долю нужно отдельно контролировать.
- Число известных объектов равно 11, поэтому результаты по фолдам очень шумные.
- Нельзя выбирать конфигурацию по одному лучшему фолду или только по среднему значению.
- При равных score на границе top-K используются `found_min`, `found_max`, `found_expected`.
- Сумма WoE предполагает аддитивность признаков и может double-count correlated geological signals.
- WoE score — ranking score, не вероятность месторождения.

### Что смотреть после запуска

1. Сравнить конфигурации по 1%, 5% и 10% площади.
2. Посмотреть `found_expected` вместе с фактическим `test_deposits`.
3. Проверить `n_unique_scores`: малое число уникальных значений означает много ничьих.
4. Оценить разброс Success Rate между фолдами.
5. Не переносить выбранные параметры на внешнюю область без отдельного zero-shot/adaptation протокола.

In [16]:
# Единый экспорт результатов WoE sensitivity

from pathlib import Path

COMPARISON_DIR = ROOT / "data" / "processed" / "woe_comparison"
COMPARISON_DIR.mkdir(parents=True, exist_ok=True)

sensitivity_export = cv.copy()

sensitivity_export["model"] = "WoE sensitivity"
sensitivity_export["features"] = ", ".join(FEATURES)
sensitivity_export["spatial_cv"] = True

sensitivity_export = sensitivity_export.rename(columns={
    "precision_at_k_expected": "precision_at_k",
    "success_rate_expected": "success_rate",
})

common_columns = [
    "model", "features", "spatial_cv",
    "n_bins", "alpha", "fold_id",
    "area_fraction", "top_k_cells",
    "precision_at_k", "success_rate",
    "found_expected", "test_deposits",
    "test_cells", "n_unique_scores",
]

sensitivity_export = sensitivity_export[
    [c for c in common_columns if c in sensitivity_export.columns]
]

sensitivity_export.to_csv(
    COMPARISON_DIR / "sensitivity_6features.csv",
    index=False,
    encoding="utf-8-sig"
)

print("Экспортировано:", COMPARISON_DIR / "sensitivity_6features.csv")
display(sensitivity_export.head())

Экспортировано: d:\Uni\Сборники\3 курс\Командная разработка\Удалённый репозиторий\data\processed\woe_comparison\sensitivity_6features.csv


,model,features,spatial_cv,n_bins,alpha,fold_id,area_fraction,top_k_cells,precision_at_k,success_rate,found_expected,test_deposits,test_cells,n_unique_scores
0,WoE sensitivity,"elevation, magnetic, gravity, dist_fault_km, d...",True,3,0.1,0,0.01,49,0.000000,0.0,0.0,1,4874,344
1,WoE sensitivity,"elevation, magnetic, gravity, dist_fault_km, d...",True,3,0.1,0,0.05,244,0.000000,0.0,0.0,1,4874,344
2,WoE sensitivity,"elevation, magnetic, gravity, dist_fault_km, d...",True,3,0.1,0,0.10,488,0.002049,1.0,1.0,1,4874,344
3,WoE sensitivity,"elevation, magnetic, gravity, dist_fault_km, d...",True,3,0.1,1,0.01,73,0.000000,0.0,0.0,2,7273,460
4,WoE sensitivity,"elevation, magnetic, gravity, dist_fault_km, d...",True,3,0.1,1,0.05,364,0.000000,0.0,0.0,2,7273,460


In [18]:
# Сборка общей сравнительной таблицы WoE

from pathlib import Path
import pandas as pd

COMPARISON_DIR = ROOT / "data" / "processed" / "woe_comparison"

FILES = [
    "baseline_3features.csv",
    "geological_6features.csv",
    "sensitivity_6features.csv",
]

# Проверяем, что все три эксперимента уже сохранены
missing_files = [
    name for name in FILES
    if not (COMPARISON_DIR / name).exists()
]

if missing_files:
    raise FileNotFoundError(
        "Не найдены результаты следующих экспериментов:\n"
        + "\n".join(missing_files)
        + "\nСначала запустите соответствующие WoE-ноутбуки."
    )

# Объединяем результаты
frames = [
    pd.read_csv(COMPARISON_DIR / name)
    for name in FILES
]

comparison = pd.concat(frames, ignore_index=True)

# Приводим числовые поля к числовому типу
numeric_columns = [
    "n_bins", "alpha", "fold_id", "area_fraction",
    "top_k_cells", "precision_at_k", "success_rate",
    "found_expected", "test_deposits", "test_cells",
    "n_unique_scores",
]

for col in numeric_columns:
    if col in comparison.columns:
        comparison[col] = pd.to_numeric(
            comparison[col], errors="coerce"
        )

comparison = comparison.sort_values(
    ["model", "n_bins", "alpha", "area_fraction", "fold_id"]
)

# Сводка по фолдам для каждой конфигурации
group_columns = [
    "model", "features", "n_bins", "alpha", "area_fraction"
]

summary = (
    comparison
    .groupby(group_columns, dropna=False)
    .agg(
        mean_precision_at_k=("precision_at_k", "mean"),
        std_precision_at_k=("precision_at_k", "std"),
        mean_success_rate=("success_rate", "mean"),
        std_success_rate=("success_rate", "std"),
        mean_found_expected=("found_expected", "mean"),
        total_test_deposits=("test_deposits", "sum"),
        folds=("fold_id", "nunique"),
    )
    .reset_index()
)

# Сохраняем CSV
detailed_path = COMPARISON_DIR / "woe_comparison_detailed.csv"
summary_path = COMPARISON_DIR / "woe_comparison_summary.csv"

comparison.to_csv(
    detailed_path, index=False, encoding="utf-8-sig"
)

summary.to_csv(
    summary_path, index=False, encoding="utf-8-sig"
)



print("Сравнение успешно сформировано!")
print("Подробные результаты:", detailed_path)
print("Сводная таблица:", summary_path)


display(summary)

Сравнение успешно сформировано!
Подробные результаты: d:\Uni\Сборники\3 курс\Командная разработка\Удалённый репозиторий\data\processed\woe_comparison\woe_comparison_detailed.csv
Сводная таблица: d:\Uni\Сборники\3 курс\Командная разработка\Удалённый репозиторий\data\processed\woe_comparison\woe_comparison_summary.csv


,model,features,n_bins,alpha,area_fraction,mean_precision_at_k,std_precision_at_k,mean_success_rate,std_success_rate,mean_found_expected,total_test_deposits,folds
0,WoE baseline,"elevation, magnetic, gravity",3,0.5,0.01,0.000000,0.000000,0.000000,0.000000,0.000000,11,5
1,WoE baseline,"elevation, magnetic, gravity",3,0.5,0.05,0.000000,0.000000,0.000000,0.000000,0.000000,11,5
2,WoE baseline,"elevation, magnetic, gravity",3,0.5,0.10,0.000016,0.000036,0.005571,0.012458,0.027857,11,5
3,WoE baseline,"elevation, magnetic, gravity",5,0.5,0.01,0.000000,0.000000,0.000000,0.000000,0.000000,11,5
4,WoE baseline,"elevation, magnetic, gravity",5,0.5,0.05,0.000000,0.000000,0.000000,0.000000,0.000000,11,5
5,WoE baseline,"elevation, magnetic, gravity",5,0.5,0.10,0.000000,0.000000,0.000000,0.000000,0.000000,11,5
6,WoE geological,"elevation, magnetic, gravity, dist_fault_km, d...",3,0.5,0.01,NaN,NaN,NaN,NaN,0.200000,11,5
7,WoE geological,"elevation, magnetic, gravity, dist_fault_km, d...",3,0.5,0.05,NaN,NaN,NaN,NaN,0.400000,11,5
8,WoE geological,"elevation, magnetic, gravity, dist_fault_km, d...",3,0.5,0.10,NaN,NaN,NaN,NaN,0.451948,11,5
9,WoE geological,"elevation, magnetic, gravity, dist_fault_km, d...",5,0.5,0.01,NaN,NaN,NaN,NaN,0.000000,11,5
